In [ ]:
# based on https://github.com/CamDavidsonPilon/Probabilistic-Programming-and-Bayesian-Methods-for-Hackers/blob/master/Chapter1_Introduction/

In [ ]:
%matplotlib inline
from IPython.core.pylabtools import figsize
import numpy as np
from matplotlib import pyplot as plt
import pymc3 as pm

In [ ]:
import json
import matplotlib
from cycler import cycler

s = json.load(open("./styles/bmh_matplotlibrc.json"))

colors = s.pop("axes.prop_cycle_colors")
s["axes.prop_cycle"] = cycler("color", colors)

matplotlib.rcParams.update(s)

Text messages per day

In [ ]:
figsize(12.5, 3.5)

count_data = np.loadtxt("data/txtdata.csv")
n_count_data = len(count_data)

plt.bar(np.arange(n_count_data), count_data, color="#348ABD")
plt.xlabel("Time (days)")
plt.ylabel("Text messages received")
plt.title("Did the user's texting habits change over time?")
plt.xlim(0, n_count_data);

**Problem:** Did the user's texts/day change significantly over time?

For starters, assume the user's habits may have changed only after one day within this observed period.

If we also assume that $C_i$, number of text messages the user sends per day $i$, follows a Poisson distribution then we can define the following:

$$
C_i \sim Poi(\lambda) \\

\lambda (t) = \begin{cases}
  \lambda_1 & \text{if } t < \tau \\
  \lambda_2 & \text{if } t \geq \tau
\end{cases}
$$

where habits may have changed signficantly after day $\tau$

Define the prior distributions for  $\lambda_1$, $\lambda_2$, and $\tau$

- $\lambda$ s can be any positive number. So exponential distribution
- Note: Exponenital distributions are sometimes defined by a parameter $\lambda$ - THIS IS NOT the same as the $\lambda (t)$ we are using as a PARAMETER FOR THE POISSON DISTRIBUTION
- So instead let's call the parameter defining the exponential distribution $\alpha$

$$
\begin{align}
&\lambda_1 \sim \text{Exp}( \alpha ) \\\
&\lambda_2 \sim \text{Exp}( \alpha )
\end{align}
$$

- $\tau$ can also be any positive number but a more refined assumption can be that we know the change in user texting behavior change somewhere between 0 and 70 days.

$$
\begin{align}
& \tau \sim \text{DiscreteUniform(1,70) }\\\\
& \Rightarrow P( \tau = k ) = \frac{1}{70}
\end{align}
$$

In [ ]:
with pm.Model() as model:
    alpha = 1/count_data.mean()

    # stochastic variables names ("lambda_1", "tau")
    lambda_1 = pm.Exponential("lambda_1", alpha) 
    lambda_2 = pm.Exponential("lambda_2", alpha)

    tau = pm.DiscreteUniform("tau", lower=0, upper= n_count_data - 1)

Here variables like `lambda_1` are random variable objects — they each represent a distribution, with no numbers actually drawn yet. 

When PyMC "samples from a random variable", it's running an inference algorithm (usually MCMC) that draws values from it.

So two phrasings are consistent:

"sampling from the exponential distribution" = the action
"sampling from the random variable `lambda_1`" = same action, but the RV object is the distribution — it's just OOP-style encapsulation

The mental model: a random variable is a named handle for a distribution. Sampling from the variable and sampling from the distribution it describes are the same thing.

Then define the function $\lambda (t)$ as defined above.

In [ ]:
with model:
    idx = np.arange(n_count_data)
    lambda_ = pm.math.switch(tau > idx, lambda_1, lambda_2)


Now associate the function $\lambda (t)$ to the observed data (`count_data`)

In [ ]:
with model:
    # notice we now define an "observed parameter"
    observation = pm.Poisson("obs", lambda_, observed=count_data)
    

Without `observed=...` set, PyMC treats the above like another prior/random variable to sample *from*.

With `observed=...`, PyMC will now use the Poisson probability mass function to compute the probability of observed text messages $C_i = k$ for each day in the observed period during MCMC:

1. **Propose:** MCMC proposes candidate values ($\lambda_1$, $\lambda_2$, and $\tau$)
2. **Score:** PyMC evaluates the Poisson PMF for every day using those candidates, multiplying all 74 probabilities together into one likelihood score (or summing log-likelihod, same thing)
3. **Keep or discard sample:** Accept or reject high score = keep the sample; low score = likely discard
4. **Loop** back to step 1. 

Repeat 15,000 times (5k tune + 10k sample). The 10k accepted samples are your posterior samples.

In [ ]:
with model:
    step = pm.Metropolis()
    trace = pm.sample(10000, tune=5000, step=step, return_inferencedata=False)

PyMC default is 4 Markov Chains

MCMC stores the posterior samples in `trace`

In [ ]:
lambda_1_samples = trace['lambda_1']
lambda_2_samples = trace['lambda_2']
tau_samples = trace['tau']

We can then visualize the histogram of these sample values and see the posterior distribution.

In [ ]:
figsize(12.5, 10)
#histogram of the samples:

ax = plt.subplot(311)
ax.set_autoscaley_on(False)

plt.hist(lambda_1_samples, histtype='stepfilled', bins=30, alpha=0.85,
         label=r"posterior of $\lambda_1$", color="#A60628", density=True)
plt.legend(loc="upper left")
plt.title(r"""Posterior distributions of the variables $\lambda_1,\;\lambda_2,\;\tau$""")
plt.xlim([15, 30])
plt.xlabel(r"$\lambda_1$ value")

ax = plt.subplot(312)
ax.set_autoscaley_on(False)
plt.hist(lambda_2_samples, histtype='stepfilled', bins=30, alpha=0.85,
         label=r"posterior of $\lambda_2$", color="#7A68A6", density=True)
plt.legend(loc="upper left")
plt.xlim([15, 30])
plt.xlabel(r"$\lambda_2$ value")

plt.subplot(313)
w = 1.0 / tau_samples.shape[0] * np.ones_like(tau_samples)
plt.hist(tau_samples, bins=n_count_data, alpha=1,
         label=r"posterior of $\tau$",
         color="#467821", weights=w, rwidth=2.)
plt.xticks(np.arange(n_count_data))

plt.legend(loc="upper left")
plt.ylim([0, .75])
plt.xlim([35, len(count_data)-20])
plt.xlabel(r"$\tau$ (in days)")
plt.ylabel("probability");

Note: we can say that the above histograms are "empirical posterior" distributions. 
- Looking at posterior of $\tau$: Most likely, the change in texting behavior happened 45 days in.
- Looking at posterior of $\lambda_1$: Before 45 days, the average texts per day was around 18 texts/day
- Looking at posterior of $\lambda_2$: After 45 days, the average texts per day was around 22 texts/day
- But aside from "hard values" for these parameters, we get distributions, so we get a sense of the uncertainty that comes with these estimates.
- This is particularly useful in interpreting $\tau$ - 50% chance change happened after 45 days, but also a near 40% chance that change happened after *44 days*. 
- Moreover, even though we assumed priors for lambda to follow exponential distribution, we now see Gaussian-like distributions. Even if we make assumptions about the parametric form of a prior distribution, we are not tied to that form during computations of a posterior.

In practice, these are preferred over parametric posteriors like `Gaussian(mean, var)` because it's still possible to sample from the empircal posterior/histograms without having to then fit another distribution function to a histogram.

Next, we are going to plot the expected texts per day using the posterior distributions:

In [ ]:
figsize(12.5, 5)

# tau_samples, lambda_1_samples, lambda_2_samples contain
# N samples from the corresponding posterior distribution
N = tau_samples.shape[0] # 10,000

expected_texts_per_day = np.zeros(n_count_data)
for day in range(0, n_count_data):
    # ix is a bool index of all tau samples corresponding to
    # the switchpoint occurring prior to value of 'day'
    ix = day < tau_samples # bool array, shape (10000,)

    # Each posterior sample corresponds to a value for tau.
    # for each day, that value of tau indicates whether we're "before"
    # (in the lambda1 "regime") or
    #  "after" (in the lambda2 "regime") the switchpoint.
    # by taking the posterior sample of lambda1/2 accordingly, we can average
    # over all samples to get an expected value for lambda on that day.
    # As explained, the "message count" random variable is Poisson distributed,
    # and therefore lambda (the poisson parameter) is the expected value of
    # "message count".
    expected_texts_per_day[day] = (lambda_1_samples[ix].sum()
                                   + lambda_2_samples[~ix].sum()) / N


plt.plot(range(n_count_data), expected_texts_per_day, lw=4, color="#E24A33",
         label="expected number of text-messages received")
plt.xlim(0, n_count_data)
plt.xlabel("Day")
plt.ylabel("Expected # text-messages")
plt.title("Expected number of text-messages received")
plt.ylim(0, 60)
plt.bar(np.arange(len(count_data)), count_data, color="#348ABD", alpha=0.65,
        label="observed texts per day")

plt.legend(loc="upper left");